# Huấn Luyện Tác Tử Cấp 8: AlphaZero-inspired Self-Play & Arena Gating (Hỗ Trợ CPU & GPU)

Notebook này đóng gói toàn bộ quy trình huấn luyện tự chơi khép kín (**Self-Play $\rightarrow$ Replay Buffer Refinement $\rightarrow$ Checkpoint Arena Gating $\ge 55\%$**) của **Level 8 (`Level8AlphaZeroAgent`)**.

Bạn chỉ cần bấm **Run All** — toàn bộ các bước dưới đây sẽ tự động chạy tuần tự:
1. **Kiểm tra Phần cứng (GPU CUDA / CPU) & Trọng số Khởi tạo ấm (Warm-Start)** từ `checkpoints/best_model.pt` của **Level 5**.
2. **Cấu hình Thích ứng Tự động (`SELFPLAY_PROFILE = "auto"`) & Tự động Chạy tiếp nối (`AUTO_RESUME = True`)** từ `checkpoints/l8_alphazero_latest.pt`.
3. **Thực thi Vòng lặp 3 Pha:**
   - **Pha 1 (Self-Play):** Tự đấu với chính mình bằng cây MCTS PUCT (`eval_mode=False`, bật nhiễu Dirichlet $\alpha=0.3, \epsilon=0.25$ tại gốc, hạ nhiệt độ $\tau: 1.0 \rightarrow 0.1$ sau 20 plies), thu thập $(s_t, \boldsymbol{\pi}_t, z_t)$ vào `SelfPlayReplayBuffer`.
   - **Pha 2 (Refinement):** Cập nhật mạng thách đấu (`Challenger`) bằng hàm mất mát `DualHeadLoss` (Soft Cross-Entropy + Value MSE).
   - **Pha 3 (Arena Gating):** `Challenger` đấu đối kháng đổi màu Trắng/Đen với `Champion` trên bộ khai cuộc cân bằng `openings_50.epd` ở chế độ `eval_mode=True`. Chỉ khi tỷ lệ điểm đạt **$\ge 55\%$**, `Challenger` mới đăng quang thành `Champion` mới (`checkpoints/l8_alphazero_best.pt`).
4. **Trực quan hóa Kết quả:** Vẽ bộ 3 biểu đồ chuẩn báo cáo (`Refinement Loss`, `Arena Gating Score Rate vs 55% Threshold`, `Replay Buffer & Champion Version`) và lưu ảnh `results/figures/fig_level8_selfplay_gating.png`.
5. **Kiểm thử Thực tế `Level8AlphaZeroAgent`:** Chạy thử tìm kiếm PUCT với mô hình vô địch và so sánh trực tiếp phân phối tiên nghiệm $P(s, a)$ của mạng với phân phối tìm kiếm $\boldsymbol{\pi}_{\text{MCTS}}$.

## 1. Thiết Lập Đường Dẫn & Kiểm Tra Phần Cứng / Checkpoint Warm-Start (Level 5)

In [ ]:
import json
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
WARM_START_CKPT = CHECKPOINT_DIR / "best_model.pt"
L8_BEST_CKPT = CHECKPOINT_DIR / "l8_alphazero_best.pt"
L8_LATEST_CKPT = CHECKPOINT_DIR / "l8_alphazero_latest.pt"
TRAINING_OUT_DIR = PROJECT_ROOT / "results" / "training"
METRICS_JSON = TRAINING_OUT_DIR / "selfplay_metrics.json"
OPENINGS_EPD = PROJECT_ROOT / "resources" / "openings_50.epd"

print(f"Thư mục gốc dự án : {PROJECT_ROOT}")
print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU      : {torch.cuda.get_device_name(0)}")
else:
    print("Thiết bị hiện tại : CPU")

if WARM_START_CKPT.exists():
    l5_meta = torch.load(WARM_START_CKPT, map_location="cpu", weights_only=False)
    print(f"\n[WARM-START SẴN SÀNG] Tìm thấy checkpoint Level 5 tại: {WARM_START_CKPT}")
    if isinstance(l5_meta, dict):
        print(
            f"  - Epoch Level 5 : {l5_meta.get('epoch')} | "
            f"Val Top-1 Acc: {l5_meta.get('val_policy_acc', 0)*100:.2f}% | "
            f"Val Value MSE: {l5_meta.get('val_value_mse', 0):.4f}"
        )
else:
    print(f"\n[CẢNH BÁO] Chưa tìm thấy {WARM_START_CKPT}. Hệ thống sẽ khởi tạo mạng từ đầu (Tabula Rasa).")

## 2. Cấu Hình Vòng Lặp Self-Play & Arena Gating (Tự Động Thích Ứng GPU / CPU)

Chế độ `SELFPLAY_PROFILE` mặc định được đặt là `"auto"`:
- **Khi chạy trên GPU (`cuda` - chế độ `"balanced_gpu"`):**
  - `ITERATIONS = 5` vòng lặp.
  - Mỗi vòng: Tự chơi `10 ván` (`50 simulations/nước`, `max_plies = 90`) $\rightarrow$ Cập nhật `25 bước mini-batch` (`batch_size = 128`) $\rightarrow$ Đấu loại Arena Gating `10 ván` (5 cặp khai cuộc $\times$ 2 màu Trắng/Đen).
  - **Tổng thời gian dự kiến trên RTX 3050:** ~8 – 12 phút cho toàn bộ 5 vòng lặp.
- **Khi chạy trên CPU (`cpu` - chế độ `"balanced_cpu"`):**
  - `ITERATIONS = 3` vòng lặp, Tự chơi `6 ván/vòng` (`25 simulations/nước`, `max_plies = 70`), Arena Gating `6 ván/vòng` (~8 – 12 phút trên CPU).
- Ngoài ra bạn có thể đổi `SELFPLAY_PROFILE = "full"` (cấu hình lớn) hoặc `"quick_test"` (~1 phút để test nhanh).

In [ ]:
SELFPLAY_PROFILE = "auto"   # Chọn: "auto" | "balanced_gpu" | "balanced_cpu" | "full" | "quick_test"
AUTO_RESUME = False          # True: tự động nối tiếp từ checkpoints/l8_alphazero_latest.pt nếu đã chạy dở
CUSTOM_ITERATIONS = None    # Đặt một số nguyên (ví dụ 8) nếu muốn ghi đè tổng số Iterations

has_gpu = torch.cuda.is_available()
effective_profile = (
    ("balanced_gpu" if has_gpu else "balanced_cpu")
    if SELFPLAY_PROFILE == "auto"
    else SELFPLAY_PROFILE
)

if effective_profile == "balanced_gpu":
    ITERATIONS = CUSTOM_ITERATIONS or 5
    GAMES_PER_ITER = 10
    MCTS_SIMS = 50
    ARENA_GAMES = 10
    TRAIN_STEPS = 25
    BATCH_SIZE = 128
    MAX_PLIES = 90
elif effective_profile == "balanced_cpu":
    ITERATIONS = CUSTOM_ITERATIONS or 3
    GAMES_PER_ITER = 6
    MCTS_SIMS = 25
    ARENA_GAMES = 6
    TRAIN_STEPS = 20
    BATCH_SIZE = 64
    MAX_PLIES = 70
elif effective_profile == "full":
    ITERATIONS = CUSTOM_ITERATIONS or 5
    GAMES_PER_ITER = 25
    MCTS_SIMS = 100
    ARENA_GAMES = 20
    TRAIN_STEPS = 40
    BATCH_SIZE = 128
    MAX_PLIES = 120
elif effective_profile == "quick_test":
    ITERATIONS = CUSTOM_ITERATIONS or 2
    GAMES_PER_ITER = 2
    MCTS_SIMS = 12
    ARENA_GAMES = 4
    TRAIN_STEPS = 5
    BATCH_SIZE = 32
    MAX_PLIES = 40
else:
    raise ValueError(f"SELFPLAY_PROFILE không hợp lệ: {SELFPLAY_PROFILE}")

resume_ckpt_path = None
if AUTO_RESUME and L8_LATEST_CKPT.exists():
    saved_l8 = torch.load(L8_LATEST_CKPT, map_location="cpu", weights_only=False)
    done_iter = int(saved_l8.get("iteration", 0))
    champ_ver = int(saved_l8.get("champion_version", 0))
    resume_ckpt_path = str(L8_LATEST_CKPT)
    if CUSTOM_ITERATIONS is None and done_iter >= ITERATIONS:
        # Mỗi lần nhấn Run All tiếp theo sẽ tự động huấn luyện nối tiếp thêm 1 đợt ITERATIONS vòng mới!
        ITERATIONS = done_iter + ITERATIONS
    print(f"[RESUME] Tìm thấy checkpoint Level 8 tại {L8_LATEST_CKPT} (đã hoàn tất {done_iter} vòng, Champion v{champ_ver}).")
    print(f"-> Sẽ lấy Champion v{champ_ver} & Replay Buffer cũ để huấn luyện tiếp từ Iteration {done_iter + 1} đến {ITERATIONS}!")

print("\n=== CẤU HÌNH HUẤN LUYỆN LEVEL 8 (ALPHAZERO SELF-PLAY) ===")
print(f"  - Chế độ (Profile)     : {effective_profile} ({'GPU CUDA' if has_gpu else 'CPU'})")
print(f"  - Tổng số Iterations   : {ITERATIONS}")
print(f"  - Số ván Self-Play/Iter: {GAMES_PER_ITER} ván ({MCTS_SIMS} sims/nước, max_plies={MAX_PLIES})")
print(f"  - Số bước Train/Iter   : {TRAIN_STEPS} steps (batch_size={BATCH_SIZE})")
print(f"  - Số ván Arena Gating  : {ARENA_GAMES} ván (ngưỡng thăng cấp >= 55%)")
print(f"  - Chạy tiếp (Resume)   : {resume_ckpt_path if resume_ckpt_path else 'Không (Khởi chạy từ Iteration 1)'}")

## 3. Thực Thi Chu Trình Self-Play $\rightarrow$ Refinement $\rightarrow$ Arena Gating
- Sau **mỗi Iteration**, hệ thống tự động lưu:
  - `checkpoints/l8_alphazero_best.pt` (Trọng số Champion tốt nhất vượt qua cổng Gating $\ge 55\%$)
  - `checkpoints/l8_alphazero_latest.pt` (Trạng thái Iteration + Replay Buffer để có thể dừng và chạy tiếp bất cứ lúc nào)
  - `results/training/selfplay_metrics.json` (Báo cáo chi tiết từng vòng lặp)

In [ ]:
from chess_ai.training.selfplay import run_selfplay_pipeline
from chess_ai.utils.epd import load_epd_file

opening_fens = [rec.fen for rec in load_epd_file(str(OPENINGS_EPD))] if OPENINGS_EPD.exists() else None

selfplay_report = run_selfplay_pipeline(
    config_path=str(PROJECT_ROOT / "configs" / "search_params.yaml"),
    warm_start_checkpoint=str(WARM_START_CKPT),
    output_dir=str(TRAINING_OUT_DIR),
    checkpoint_dir=str(CHECKPOINT_DIR),
    iterations=ITERATIONS,
    selfplay_games_override=GAMES_PER_ITER,
    mcts_sims_override=MCTS_SIMS,
    arena_games_override=ARENA_GAMES,
    train_steps_per_iter=TRAIN_STEPS,
    batch_size=BATCH_SIZE,
    learning_rate=3e-4,
    max_plies=MAX_PLIES,
    resume_path=resume_ckpt_path,
    initial_fens=opening_fens,
    seed=42,
    verbose=True,
)

## 4. Trực Quan Hóa Kết Quả Self-Play & Checkpoint Arena Gating
Đọc tệp `results/training/selfplay_metrics.json`, tổng hợp bảng kết quả qua từng Iteration và xuất biểu đồ vào `results/figures/fig_level8_selfplay_gating.png`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if not METRICS_JSON.exists():
    raise FileNotFoundError(f"Chưa tìm thấy tệp kết quả Self-Play tại: {METRICS_JSON}")

sp_data = json.loads(METRICS_JSON.read_text(encoding="utf-8"))
rows = []
for item in sp_data.get("history", []):
    ar = item.get("arena", {})
    rows.append({
        "iteration": item["iteration"],
        "selfplay_games": item["selfplay_games"],
        "samples_added": item["samples_added"],
        "buffer_size": item["buffer_size"],
        "total_loss": item["train_total_loss"],
        "policy_loss": item["train_policy_loss"],
        "value_loss": item["train_value_loss"],
        "arena_W_D_L": f"{ar.get('wins',0)}W-{ar.get('draws',0)}D-{ar.get('losses',0)}L",
        "arena_score_pct": round(ar.get("score_rate", 0.0) * 100.0, 2),
        "accepted": ar.get("accepted", False),
        "champion_version": item["champion_version"],
        "elapsed_sec": item["elapsed_sec"],
    })

sp_df = pd.DataFrame(rows)
if not sp_df.empty:
    display(sp_df)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    # Biểu đồ 1: Refinement Loss trên Replay Buffer
    axes[0].plot(sp_df["iteration"], sp_df["total_loss"], marker="o", label="Total Loss", color="#2563eb")
    axes[0].plot(sp_df["iteration"], sp_df["policy_loss"], marker="s", label="Soft Policy Loss", color="#9333ea")
    axes[0].plot(sp_df["iteration"], sp_df["value_loss"], marker="^", label="Value MSE Loss", color="#0284c7")
    axes[0].set_title("1. Self-Play Replay Buffer Losses")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Loss")
    axes[0].set_xticks(sp_df["iteration"])
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Biểu đồ 2: Arena Gating Score Rate (%) vs Ngưỡng 55%
    bar_colors = ["#16a34a" if acc else "#64748b" for acc in sp_df["accepted"]]
    axes[1].bar(sp_df["iteration"], sp_df["arena_score_pct"], color=bar_colors, alpha=0.85, width=0.55, label="Challenger Score (%)")
    axes[1].axhline(55.0, linestyle="--", color="#dc2626", linewidth=1.8, label="Gating Threshold (55%)")
    axes[1].axhline(50.0, linestyle=":", color="#94a3b8", linewidth=1.2, label="Even Match (50%)")
    axes[1].set_title("2. Checkpoint Arena Gating Score Rate (%)")
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Challenger Score Rate (%)")
    axes[1].set_ylim(0, 105)
    axes[1].set_xticks(sp_df["iteration"])
    axes[1].grid(True, axis="y", alpha=0.3)
    axes[1].legend()

    # Biểu đồ 3: Kích thước Replay Buffer & Phiên bản Champion
    ax3_twin = axes[2].twinx()
    l1 = axes[2].plot(sp_df["iteration"], sp_df["buffer_size"], marker="o", color="#0d9488", label="Replay Buffer Size")
    l2 = ax3_twin.step(sp_df["iteration"], sp_df["champion_version"], where="post", marker="*", markersize=9, color="#ea580c", label="Champion Version")
    axes[2].set_title("3. Replay Buffer Growth & Champion Promotions")
    axes[2].set_xlabel("Iteration")
    axes[2].set_ylabel("Replay Buffer Samples", color="#0d9488")
    ax3_twin.set_ylabel("Champion Version", color="#ea580c")
    axes[2].set_xticks(sp_df["iteration"])
    axes[2].grid(True, alpha=0.3)
    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    axes[2].legend(lines, labels, loc="upper left")

    plt.tight_layout()
    fig_dir = PROJECT_ROOT / "results" / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    out_fig = fig_dir / "fig_level8_selfplay_gating.png"
    plt.savefig(out_fig, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu biểu đồ Self-Play & Arena Gating tại: {out_fig}")

## 5. Kiểm Thử Thực Tế `Level8AlphaZeroAgent` Với Checkpoint Vô Địch
Nạp `checkpoints/l8_alphazero_best.pt` vào `Level8AlphaZeroAgent` và chạy thử tìm kiếm PUCT trên các thế cờ khai cuộc & chiến thuật.

In [ ]:
import chess
from chess_ai.agents.level8_alphazero import Level8AlphaZeroAgent
from chess_ai.core.chess_env import ChessEnv

agent8 = Level8AlphaZeroAgent(checkpoint_path=str(L8_BEST_CKPT), simulations=100)
print(f"Đã nạp checkpoint Level 8: {agent8.checkpoint_loaded} ({agent8.checkpoint_path})")
print(f"Metadata checkpoint      : {agent8.checkpoint_meta}")

test_fens = [
    ("Khai cuộc xuất phát chuẩn", chess.STARTING_FEN),
    ("Thế cờ chiến thuật WAC.001 (Tìm nước Qg6)", "2rr3k/pp3pp1/1nnqbN1p/3pN3/2pP4/2P3Q1/PPB4P/R4RK1 w - - 0 1"),
]

for title, fen in test_fens:
    env = ChessEnv(fen=fen)
    move = agent8.select_move(env, max_workload=100)
    st = agent8.last_search_stats
    print(f"\n[{title}]")
    print(f"  - Nước đi Level 8 chọn: {env.board.san(move)} ({move.uci()})")
    print(f"  - Thống kê tìm kiếm   : sims={int(st['simulations'])}, max_depth={int(st['depth'])}, Q_eval={st['eval']:+.3f}, root_V={st['root_value']:+.3f}, time={st['time_sec']:.2f}s")